In [4]:
import pandas as pd
import numpy as np

In [5]:
###loading dataset

df = pd.read_csv("Titanic-Dataset.csv")

In [7]:
#inspecting first few rows

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [10]:
print("Dimensions", df.shape)

df.info()

df.describe()

df.describe(include=['object'])
      

Dimensions (891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.9 KB


C:\Users\91877\AppData\Local\Temp\ipykernel_16100\2901821649.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include=['object'])


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [11]:
# 1. Fill missing 'Embarked' with the mode 
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 2. Fill missing 'Age' using the median age per Sex
df['Age'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(lambda x: x.fillna(x.median()))

# 3. over 75% is missing, extract deck level or flag if cabin is present
df['Has_Cabin'] = df['Cabin'].notna().astype(int)
df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

# Check missing values again to confirm
print(df.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Has_Cabin        0
Deck             0
dtype: int64


In [13]:
# 1. Family Size: Combine SibSp (siblings/spouses) and Parch (parents/children)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 2. IsAlone flag
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Extract Titles from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles together
rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']
df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# 4. Binning Age and Fare into categories
df['AgeGroup'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Child', 'Teen', 'Adult', 'Middle-Aged', 'Senior'])
df['FareBin'] = pd.qcut(df['Fare'], q=4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])

In [14]:
overall_survival = df['Survived'].mean()
print(f"Overall Survival Rate: {overall_survival:.2%}")

Overall Survival Rate: 38.38%


In [15]:
sex_survival = df.groupby('Sex')['Survived'].agg(['count', 'mean'])
sex_survival.columns = ['Total Passengers', 'Survival Rate']
print(sex_survival)

        Total Passengers  Survival Rate
Sex                                    
female               314       0.742038
male                 577       0.188908


In [16]:
pclass_survival = df.groupby('Pclass')['Survived'].agg(['count', 'mean'])
pclass_survival.columns = ['Total Passengers', 'Survival Rate']
print(pclass_survival)

        Total Passengers  Survival Rate
Pclass                                 
1                    216       0.629630
2                    184       0.472826
3                    491       0.242363


In [17]:
pivot_table = df.pivot_table(index='Pclass', columns='Sex', values='Survived', aggfunc='mean')
print(pivot_table)

Sex       female      male
Pclass                    
1       0.968085  0.368852
2       0.921053  0.157407
3       0.500000  0.135447


In [18]:
print(df.groupby('IsAlone')['Survived'].mean())
print(df.groupby('FamilySize')['Survived'].mean())

IsAlone
0    0.505650
1    0.303538
Name: Survived, dtype: float64
FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64
